In [1]:
pip install pandas

  Using cached pandas-3.0.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.4-cp311-cp311-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.0.2-cp311-cp311-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.4-cp311-cp311-macosx_14_0_arm64.whl (5.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]━━━━ 1/2 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import sqlite3

# 读取 CSV
df = pd.read_csv("spotify_2014_2020_dataset.csv")
print(df.shape)
print(df.head())

(1701, 7)
                 Track_ID      Track_Name  Duration_Min  \
0  6dBUzqjtbnIa1TwYbyw5CM     Lovers Rock          3.57   
1  5J4ZkQpzMUFojo1CtAZYpn  Love Me Harder          3.94   
2  2e3Ea0o24lReQFR4FA7yXH      Love Yourz          3.52   
3  6nGeLlakfzlBcFdZXteDq7      Love Story          5.27   
4  5BJSZocnCeSNeYMj3iVqM7   Love Runs Out          3.74   

                     Artists Release_Date  Year Query  
0                    TV Girl     6/5/2014  2014  love  
1  Ariana Grande, The Weeknd    8/22/2014  2014  love  
2                    J. Cole    12/9/2014  2014  love  
3                     Indila     1/1/2014  2014  love  
4                OneRepublic     1/1/2014  2014  love  


In [2]:
# 清洗：解析日期，去掉首尾空格
df["Release_Date"] = pd.to_datetime(df["Release_Date"], format="%m/%d/%Y", errors="coerce")
df["Track_Name"] = df["Track_Name"].str.strip()
df["Artists"] = df["Artists"].str.strip()
df["Query"] = df["Query"].str.strip()

# 检查空值
print("空值统计：")
print(df.isnull().sum())

空值统计：
Track_ID         0
Track_Name       0
Duration_Min     0
Artists          0
Release_Date    10
Year             0
Query            0
dtype: int64


In [3]:
# 创建 SQLite 数据库，把 df 写进去
con = sqlite3.connect("spotify.db")
df.to_sql("tracks", con, if_exists="replace", index=False)
print("写入完成，验证一下：")

result = pd.read_sql("SELECT COUNT(*) AS total FROM tracks", con)
print(result)

写入完成，验证一下：
   total
0   1701


In [4]:
# Q1: 每年歌曲数量
q1 = pd.read_sql("""
    SELECT Year, COUNT(*) AS track_count
    FROM tracks
    GROUP BY Year
    ORDER BY Year
""", con)

print(q1)

   Year  track_count
0  2014          299
1  2015          296
2  2016          303
3  2017          275
4  2018          207
5  2019          205
6  2020          116


In [5]:
# Q2: CTE — 每年各 mood 占比
q2 = pd.read_sql("""
    WITH year_totals AS (
        SELECT Year, COUNT(*) AS total
        FROM tracks
        GROUP BY Year
    )
    SELECT t.Year,
           t.Query AS mood,
           COUNT(*) AS mood_count,
           ROUND(COUNT(*) * 100.0 / yt.total, 1) AS pct_of_year
    FROM tracks t
    JOIN year_totals yt ON t.Year = yt.Year
    GROUP BY t.Year, t.Query
    ORDER BY t.Year, pct_of_year DESC
""", con)

print(q2)

    Year   mood  mood_count  pct_of_year
0   2014    sad          80         26.8
1   2014   rock          68         22.7
2   2014  remix          58         19.4
3   2014  night          47         15.7
4   2014   love          46         15.4
5   2015    sad          72         24.3
6   2015   rock          68         23.0
7   2015  remix          60         20.3
8   2015  night          51         17.2
9   2015   love          45         15.2
10  2016    sad          68         22.4
11  2016   rock          65         21.5
12  2016  remix          65         21.5
13  2016  night          58         19.1
14  2016   love          47         15.5
15  2017   rock          72         26.2
16  2017    sad          64         23.3
17  2017  remix          60         21.8
18  2017  night          42         15.3
19  2017   love          37         13.5
20  2018    sad          51         24.6
21  2018   rock          45         21.7
22  2018  remix          44         21.3
23  2018   love 

In [6]:
# Q3: Window Function — 累计歌曲数
q3 = pd.read_sql("""
    SELECT Year,
           COUNT(*) AS tracks_this_year,
           SUM(COUNT(*)) OVER (ORDER BY Year) AS running_total
    FROM tracks
    GROUP BY Year
    ORDER BY Year
""", con)

print(q3)

   Year  tracks_this_year  running_total
0  2014               299            299
1  2015               296            595
2  2016               303            898
3  2017               275           1173
4  2018               207           1380
5  2019               205           1585
6  2020               116           1701


In [7]:
# Q4: Window Function — 每个 mood 内按时长排名
q4 = pd.read_sql("""
    SELECT Track_Name,
           Artists,
           Query AS mood,
           Duration_Min,
           RANK() OVER (PARTITION BY Query ORDER BY Duration_Min DESC) AS rank_in_mood
    FROM tracks
    ORDER BY mood, rank_in_mood
    LIMIT 20
""", con)

print(q4)

                                           Track_Name  \
0                                            Lovesong   
1                                        Love Tonight   
2                                      Lover Is a Day   
3   Lover, You Should've Come Over - Live At Colum...   
4                                         Love & Hate   
5      Drunk in Love Remix (feat. JAY-Z & Kanye West)   
6                                   Love, Me Normally   
7                                       Reckless Love   
8                                 Love Letters to God   
9                                             Redbone   
10                                     Love Like This   
11                        Drunk in Love (feat. JAY-Z)   
12                               Love So Great - Live   
13                                         Love Story   
14                                       Love and War   
15                                        Best Friend   
16                   Love Story

In [8]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [9]:
!pip install nbformat

In [10]:
import plotly.express as px

# 每年歌曲数量 — 柱状图
fig = px.bar(
    q1,
    x="Year",
    y="track_count",
    title="Spotify Tracks by Year (2014–2020)",
    labels={"track_count": "Number of Tracks", "Year": "Year"},
    color_discrete_sequence=["#1DB954"],
    text="track_count",
)
fig.update_traces(textposition="outside")
fig.update_layout(plot_bgcolor="white")
fig.show()

In [11]:
MOOD_COLORS = {
    "sad":   "#5B9BD5",
    "rock":  "#E06C75",
    "remix": "#98C379",
    "night": "#C678DD",
    "love":  "#E5C07B",
}

# pivot 成宽表，每个 mood 一列
df_pivot = q2.pivot(index="Year", columns="mood", values="pct_of_year").fillna(0)

import plotly.graph_objects as go

fig = go.Figure()
for mood in df_pivot.columns:
    fig.add_trace(go.Scatter(
        x=df_pivot.index,
        y=df_pivot[mood],
        mode="lines+markers",
        name=mood,
        stackgroup="one",
        line=dict(color=MOOD_COLORS.get(mood, "#888")),
    ))

fig.update_layout(
    title="Mood Distribution per Year (% of Annual Tracks)",
    xaxis_title="Year",
    yaxis_title="% of Tracks",
    plot_bgcolor="white",
)
fig.show()

In [12]:
# Top 10 艺人 — 水平柱状图
q_artists = pd.read_sql("""
    SELECT TRIM(SUBSTR(Artists, 1,
               CASE WHEN INSTR(Artists, ',') > 0
                    THEN INSTR(Artists, ',') - 1
                    ELSE LENGTH(Artists) END)) AS artist,
           COUNT(*) AS appearances
    FROM tracks
    GROUP BY artist
    ORDER BY appearances DESC
    LIMIT 10
""", con)

fig = px.bar(
    q_artists,
    x="appearances",
    y="artist",
    orientation="h",
    title="Top 10 Most Frequent Artists (2014–2020)",
    labels={"appearances": "Track Appearances", "artist": "Artist"},
    color="appearances",
    color_continuous_scale="Greens",
    text="appearances",
)
fig.update_traces(textposition="outside")
fig.update_layout(
    plot_bgcolor="white",
    yaxis={"categoryorder": "total ascending"}
)
fig.show()

In [14]:
import os
os.makedirs("tableau_data", exist_ok=True)

# 主数据表
df.to_csv("tableau_data/spotify_clean.csv", index=False)

# mood 占比（用于堆叠图）
q2.to_csv("tableau_data/mood_share_by_year.csv", index=False)

# top 艺人
q_artists.to_csv("tableau_data/top_artists.csv", index=False)

print("导出完成")

导出完成


In [15]:
import json

# Chart 1 数据
chart1 = {
    "x": q1["Year"].tolist(),
    "y": q1["track_count"].tolist()
}

# Chart 2 数据
df_pivot = q2.pivot(index="Year", columns="mood", values="pct_of_year").fillna(0)
chart2 = {}
for mood in df_pivot.columns:
    chart2[mood] = df_pivot[mood].tolist()
chart2["years"] = df_pivot.index.tolist()

# Chart 3 数据
chart3 = {
    "artists": q_artists["artist"].tolist(),
    "appearances": q_artists["appearances"].tolist()
}

print("chart1 =", json.dumps(chart1))
print()
print("chart2 =", json.dumps(chart2))
print()
print("chart3 =", json.dumps(chart3))

chart1 = {"x": [2014, 2015, 2016, 2017, 2018, 2019, 2020], "y": [299, 296, 303, 275, 207, 205, 116]}

chart2 = {"love": [15.4, 15.2, 15.5, 13.5, 18.4, 16.6, 26.7], "night": [15.7, 17.2, 19.1, 15.3, 14.0, 19.0, 26.7], "remix": [19.4, 20.3, 21.5, 21.8, 21.3, 17.6, 34.5], "rock": [22.7, 23.0, 21.5, 26.2, 21.7, 17.1, 0.0], "sad": [26.8, 24.3, 22.4, 23.3, 24.6, 29.8, 12.1], "years": [2014, 2015, 2016, 2017, 2018, 2019, 2020]}

chart3 = {"artists": ["Lil Peep", "Twenty One Pilots", "Post Malone", "Tame Impala", "Imagine Dragons", "Milky Chance", "The Weeknd", "Beyonc\u00e9", "NF", "Billie Eilish"], "appearances": [21, 17, 14, 13, 13, 12, 11, 11, 10, 10]}
